In [ ]:
from importlib.metadata import version
import tiktoken
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import utils
print("torch version:", version("torch"))

torch version: 2.10.0


In [31]:
# Reduced dataset with 100k rows for testing
cloud_url = "https://syncandshare.lrz.de/dl/fiHE8nDPcb4nww3VCn4QmN/reduced_dataset_100k.csv"
# Uncomment the following line to use the full dataset
# cloud_url = "https://syncandshare.lrz.de/dl/fiHE8nDPcb4nww3VCn4QmN/full_dataset.csv"

try:
    print("Loading dataset from cloud...")
    df = pd.read_csv(cloud_url)
    print("Dataset loaded successfully!\n")
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")

def format_csv(row):
    title = str(row['title'])
    ingredients = str(row['ingredients']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    directions = str(row['directions']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    return f"Recepie: {title}\nIngredients: {ingredients}\nDirections: {directions}"

raw_text = "".join(df.apply(format_csv, axis=1))
print(f"The entire dataset has been successfully converted into a single string.")
print(f"Total character count: {len(raw_text)}")
print("\nPreview of the first 100 characters:")
print(raw_text[:100])

Loading dataset from cloud...
Dataset loaded successfully!

The entire dataset has been successfully converted into a single string.
Total character count: 47074254

Preview of the first 100 characters:
Recepie: No-Bake Nut Cookies
Ingredients: 1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/


In [33]:
# TODO -> umstellen auf full raw text?
# For a faster computation we only select the first 100 characters of the text for the next steps
raw_text = raw_text[:1000]
print(raw_text[:1000])

Recepie: No-Bake Nut Cookies
Ingredients: 1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/2 tsp. vanilla, 1/2 c. broken nuts (pecans), 2 Tbsp. butter or margarine, 3 1/2 c. bite size shredded rice biscuits
Directions: In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine., Stir over medium heat until mixture bubbles all over top., Boil and stir 5 minutes more. Take off heat., Stir in vanilla and cereal; mix well., Using 2 teaspoons, drop and shape into 30 clusters on wax paper., Let stand until firm, about 30 minutes.Recepie: Jewell Ball'S Chicken
Ingredients: 1 small jar chipped beef, cut up, 4 boned chicken breasts, 1 can cream of mushroom soup, 1 carton sour cream
Directions: Place chipped beef on bottom of baking dish., Place chicken on top of beef., Mix soup and cream together; pour over chicken. Bake, uncovered, at 275\u00b0 for 3 hours.Recepie: Creamy Corn
Ingredients: 2 (16 oz.) pkg. frozen corn, 1 (8 oz.) pkg. cream cheese, cubed

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [ ]:
torch.manual_seed(123)

tokenizer = tiktoken.get_encoding("gpt2")
model = utils.gpt.GPTModel(GPT_CONFIG_124M)

encoded_text = tokenizer.encode(raw_text)

vocab_size = 50257
output_dim = 768
max_len = 1024
context_length = max_len


token_embedding_layer = nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

max_length = 9

# shuffle = False, da die Reihenfolge der Tokens wichtig ist zum Testen!
dataloader = utils.gpt.create_dataloader_v1(raw_text, shuffle=False, drop_last=False, batch_size=8, max_length=max_length, stride=max_length)

# TODO wofür drop_last?
# um den letzten batch wezuwerfen, wenn dieser unvollständig ist

for batch in dataloader:
    x, y = batch
    out = model(x)
    break

In [79]:
print("Input batch:\n", x)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[ 3041,   344, 21749,    25,  1400,    12,    33,   539, 11959],
        [45305,   198, 41222,    25,   352,   269,    13, 14245, 11856],
        [ 7586,  7543,    11,   352,    14,    17,   269,    13, 28959],
        [  515,  7545,    11,   352,    14,    17, 23053,    13, 16858],
        [   11,   352,    14,    17,   269,    13,  5445, 14380,   357],
        [43106,   504,   828,   362,   309, 24145,    13,  9215,   393],
        [ 6145, 34569,    11,   513,   352,    14,    17,   269,    13],
        [13197,  2546, 37624, 11464, 50128,   198, 13470,   507,    25]])

Output shape: torch.Size([8, 9, 50257])
tensor([[[-0.3729, -0.5560,  0.6856,  ..., -0.8469, -0.2462, -0.6611],
         [-0.1548, -0.4527,  0.0057,  ..., -0.2752, -0.5745, -0.2809],
         [ 0.9885, -0.1872,  0.6306,  ...,  0.3336,  0.2848,  0.4580],
         ...,
         [-0.0972,  0.0123, -0.5395,  ...,  0.1576, -0.3580, -0.4623],
         [ 0.5236, -0.2556,  0.6495,  ..., -0.5351,  0.3268, -

In [80]:
model.eval() # disable dropout

out = utils.gpt.generate_text_simple(
    model=model,
    idx=x, 
    max_new_tokens=2, 
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output:", out)
print("Output length:", len(out[0]))

Output: tensor([[ 3041,   344, 21749,    25,  1400,    12,    33,   539, 11959, 40467,
         16634],
        [45305,   198, 41222,    25,   352,   269,    13, 14245, 11856, 34348,
         45404],
        [ 7586,  7543,    11,   352,    14,    17,   269,    13, 28959, 21491,
         46463],
        [  515,  7545,    11,   352,    14,    17, 23053,    13, 16858, 33043,
         47269],
        [   11,   352,    14,    17,   269,    13,  5445, 14380,   357, 15953,
         46779],
        [43106,   504,   828,   362,   309, 24145,    13,  9215,   393, 39838,
         31887],
        [ 6145, 34569,    11,   513,   352,    14,    17,   269,    13, 49512,
         49030],
        [13197,  2546, 37624, 11464, 50128,   198, 13470,   507,    25, 49007,
          6074]])
Output length: 11


In [82]:
decoded_text = tokenizer.decode(out[0].squeeze(0).tolist())
print(decoded_text)

Recepie: No-Bake Nut inconsistenciesppo
